# Entity ***Tokens***
+ Layer bronze
+ Priority 1
---
### Imports

In [0]:
%run ../00_functions/medallion_functions

In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
dbutils.widgets.text("layer", "bronze")
dbutils.widgets.text("entity_name", "tokens")
dbutils.widgets.text("load_pattern", "2")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
PAGE_SIZE = 1000
tokens_list = []
query_variables = {
    "skip": 0
}

In [0]:
tokens_query = """ 
  query TokensQuery($skip: Int!){
    tokens(first: 1000, 
      skip: $skip,
      orderBy: volumeUSD, 
      orderDirection: desc
    ) {
      id
      symbol
      name
      decimals
      totalSupply
      volume
      volumeUSD
      feesUSD
      txCount
      poolCount
      totalValueLocked
      totalValueLockedUSD
      derivedETH
  }
}
"""

### Execution

In [0]:
while True:
    print(f"*INFO*: Skip rows amount: {query_variables["skip"]}.")
    response = requests.post(SUBGRAPH_URL, json={"query": tokens_query, "variables": query_variables})
    tokens_list.extend(response.json()["data"][entity_name])

    time.sleep(0.3)

    rows_loaded = len(response.json()["data"][entity_name])
    print(f"*INFO*: Number of rows loaded: {rows_loaded}.")

    if rows_loaded == PAGE_SIZE:
        query_variables.update({"skip": query_variables["skip"] + PAGE_SIZE}) 
    elif rows_loaded == 0: 
        break
    else:
        query_variables.update({"skip": query_variables["skip"] + rows_loaded})     

In [0]:
tokens_df = spark.createDataFrame(tokens_list)
tokens_df = tokens_df.withColumn("load_timestamp", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(tokens_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)